In [1]:
import os; os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == "notebooks" else None

# 02 - Baselines: PI controller and stored RL runs

Reproduces the PI-controller benchmark number, loads the results/learning-curve files
written by `scripts/evaluate.py` and `scripts/train.py`, and re-evaluates the best
stored RL checkpoint to check it is reproducible. Every cell that reads a file under
`data/` prints a clear message and continues (does not raise) if that file is missing -
in that case, run `bash scripts/reproduce.sh` first and re-run this notebook.

In [2]:
import json
from pathlib import Path

import pandas as pd

from rl_tokamak.env import EnvConfig, RampupEnv
from rl_tokamak.controllers import PIController
from rl_tokamak.evaluate import run_controller, run_policy
from rl_tokamak.policies import load_policy
from rl_tokamak.plotting import plot_episodes, plot_learning_curves

REPRODUCE_MSG = "run bash scripts/reproduce.sh first"

## (a) Reproduce the PI-controller number

`PIController` re-implements the paper's PI policy (Section "Policies", arXiv 2510.11283)
on a fresh environment. The ITER-hybrid TORAX dynamics are deterministic, so this run
should land within 0.01 of the published 3.79.

In [3]:
env = RampupEnv(EnvConfig())
pi_result = run_controller(env, PIController())
print("PI benchmark_return:", pi_result["benchmark_return"], " failed:", pi_result["failed"])

assert abs(pi_result["benchmark_return"] - 3.79) < 0.01, pi_result["benchmark_return"]
print("matches paper PI return (3.79) within 0.01")

PI benchmark_return: 3.79192295529996  failed: False
matches paper PI return (3.79) within 0.01


In [4]:
classical_path = Path("data/results/classical.json")
if classical_path.exists():
    classical = json.loads(classical_path.read_text())
    stored = classical["pi"]["benchmark_return"]
    print("stored data/results/classical.json PI return:", stored, " (paper", classical["pi"]["paper"], ")")
    print("this run vs stored: abs diff =", abs(pi_result["benchmark_return"] - stored))
else:
    print(f"{classical_path} not found - {REPRODUCE_MSG}")

stored data/results/classical.json PI return: 3.79192295529996  (paper 3.79 )
this run vs stored: abs diff = 0.0


## (b) Results summary table

`data/results/summary.json` (written by `scripts/evaluate.py --summary`) collects every
classical and RL result into one table: `policy`, `group`, `return`, `paper`, `env_steps`,
`minutes`, and a few end-of-episode physics numbers.

In [5]:
summary_path = Path("data/results/summary.json")
summary_df = None
if summary_path.exists():
    summary_df = pd.DataFrame(json.loads(summary_path.read_text())).sort_values("return", ascending=False)
else:
    print(f"{summary_path} not found - {REPRODUCE_MSG}")
summary_df

,policy,group,return,paper,env_steps,minutes,failed,q_min_final,fGW_max,Q_final,Ip_final_MA,t_q_min_below_1_s,audited,return_std,run,best_during_training,steps_to_beat_pi,dataset_transitions,first_hmode_proxy_s
28,PPO (SB3) (seed 1),online,48.984203,NaN,112136.0,46.017089,False,0.709860,1.228987,755.057899,10.786013,59.0,3.527899,NaN,ppo_s1,48.984203,48008.0,NaN,102.0
31,SAC (SB3) (seed 1),online,27.077518,NaN,74872.0,45.986644,False,0.911504,1.092028,283.719252,6.282144,51.0,1.994850,NaN,sac_s1,27.771992,15624.0,NaN,103.0
19,MBPO (ensemble + SAC) (seed 1),online,18.420217,NaN,1510.0,53.718154,False,0.669031,1.796170,123.224134,3.000000,99.0,1.885027,NaN,mbpo_s1,18.420217,1510.0,NaN,92.0
22,MBPO (ensemble + SAC) (seed 4),online,10.823221,NaN,3752.0,22.006786,False,1.285160,1.106340,109.798212,6.324200,0.0,2.013333,NaN,mbpo_s4,10.823221,1789.0,NaN,103.0
8,MBPO (ensemble + SAC) [obs=full] (seed 1),ablation,5.625857,NaN,2177.0,29.041628,False,0.582903,1.179515,36.619805,15.000000,79.0,3.687255,NaN,mbpo_obs-full_s1,21.289920,906.0,NaN,103.0
35,"CEM open-loop schedule search, objective: benc...",reference,4.076435,NaN,14496.0,45.811554,False,0.478996,1.200785,18.549403,14.583760,94.0,3.571965,NaN,NaN,NaN,NaN,NaN,NaN
34,TD3+BC on pi_noisy_0.3 (seed 0),offline,4.010587,NaN,0.0,18.832530,False,0.430682,1.195889,16.934398,14.821569,96.0,3.563716,NaN,td3bc_pi_noisy_0.3_s0,4.010587,NaN,3020.0,102.0
5,Behaviour cloning on pi_noisy_0.3 (seed 0),offline,3.987910,NaN,0.0,17.760489,False,0.427343,1.194523,16.070998,15.000000,98.0,3.559676,NaN,bc_pi_noisy_0.3_s0,3.987910,NaN,3020.0,102.0
24,MOPO on pi_det (seed 0),offline,3.937362,NaN,0.0,32.424704,False,0.693537,1.038702,15.894468,15.000000,59.0,2.289207,NaN,mopo_pi_det_s0,3.937362,NaN,151.0,104.0
36,"CEM open-loop schedule search, objective: audi...",reference,3.870824,NaN,24160.0,41.430459,False,0.516423,1.187062,14.152283,13.154897,93.0,3.625566,NaN,NaN,NaN,NaN,NaN,NaN


## (c) Re-evaluate the best stored RL checkpoint

Picks the online run (`algo` in `ppo`/`sac`/`mbpo`) with the highest stored
`result.json["benchmark_return"]`, reloads its policy with `load_policy`, and re-runs
one deterministic episode. The ITER-hybrid environment and every stored policy here are
deterministic, so the fresh return should match the stored one to numerical precision.

In [6]:
ONLINE_ALGOS = {"ppo", "sac", "mbpo"}

best_run, best_return = None, -float("inf")
for result_path in sorted(Path("data/runs").glob("*/result.json")):
    cfg = json.loads((result_path.parent / "config.json").read_text())
    if cfg["algo"] not in ONLINE_ALGOS:
        continue
    res = json.loads(result_path.read_text())
    if res["benchmark_return"] > best_return:
        best_run, best_return = result_path.parent, res["benchmark_return"]

if best_run is None:
    print(f"no online run result.json found under data/runs - {REPRODUCE_MSG}")
else:
    print("best online run:", best_run.name, " stored benchmark_return =", best_return)

best online run: ppo_s1  stored benchmark_return = 48.98420308313504


In [7]:
eval_env = None
if best_run is not None:
    env_cfg, policy_fn = load_policy(best_run)
    eval_env = RampupEnv(env_cfg)
    ep = run_policy(eval_env, policy_fn)
    print("re-evaluated benchmark_return:", ep["benchmark_return"])
    diff = abs(ep["benchmark_return"] - best_return)
    print("abs diff vs stored result.json:", diff)
    # Same tolerance as scripts/reproduce.sh: relative 1e-6. Runs trained on another CPU (see the run's
    # config.json "host") can differ in the last digits because XLA compiles different kernels per CPU.
    tol = 1e-6 * max(1.0, abs(best_return))
    assert diff <= tol, diff
    print(f"matches stored result.json within {tol:.1e}")
else:
    print(f"skipped - {REPRODUCE_MSG}")

re-evaluated benchmark_return: 48.98419975342531
abs diff vs stored result.json: 3.329709734600783e-06
matches stored result.json within 4.9e-05


## (d) Learning curves

`curve.json` (written during training) holds the deterministic-policy evaluation curve
for every online run. Plots all `ppo`/`sac`/`mbpo` runs found under `data/runs/` against
simulator steps used, with the PI (3.79) and open-loop (3.40) benchmark returns as
reference lines.

In [8]:
online_runs = {}
for cfg_path in sorted(Path("data/runs").glob("*/config.json")):
    cfg = json.loads(cfg_path.read_text())
    if cfg["algo"] in ONLINE_ALGOS and (cfg_path.parent / "curve.json").exists():
        online_runs[cfg_path.parent.name] = cfg_path.parent

if online_runs:
    print("online runs with a curve:", sorted(online_runs))
    fig = plot_learning_curves(online_runs, x="env_steps", references={"PI": 3.79, "open-loop": 3.40})
else:
    print(f"no online runs with curve.json found under data/runs - {REPRODUCE_MSG}")
    fig = None
fig

online runs with a curve: ['mbpo_ipfloor1MA_s0', 'mbpo_obs-full_s0', 'mbpo_obs-full_s1', 'mbpo_obs-scalars_s0', 'mbpo_obs-scalars_s1', 'mbpo_patched_s0', 'mbpo_patched_s1', 'mbpo_patched_s2', 'mbpo_reward-benchmark_s0', 'mbpo_reward-benchmark_s1', 'mbpo_reward-benchmark_s2', 'mbpo_reward-qminsafe_s0', 'mbpo_s0', 'mbpo_s1', 'mbpo_s2', 'mbpo_s3', 'mbpo_s4', 'mbpo_s5', 'ppo_s0', 'ppo_s1', 'sac_ipfloor1MA_s0', 'sac_s0', 'sac_s1']


<Figure size 700x420 with 1 Axes>

## (e) Best run vs PI trajectory

Compares the final deterministic episode of the best online run (`final_episode.csv`,
written by `evaluate_run`) against the PI-controller reference trajectory
(`data/trajectories/pi.csv`, written by `scripts/evaluate.py --classical`).

In [9]:
pi_csv = Path("data/trajectories/pi.csv")
best_episode_csv = best_run / "final_episode.csv" if best_run is not None else None

if best_episode_csv is not None and best_episode_csv.exists() and pi_csv.exists():
    fig2 = plot_episodes({best_run.name: best_episode_csv, "PI": pi_csv},
                         title=f"{best_run.name} vs PI controller")
else:
    missing = [p for p in (best_episode_csv, pi_csv) if p is None or not p.exists()]
    print(f"missing {missing} - {REPRODUCE_MSG}")
    fig2 = None
fig2

<Figure size 1300x900 with 9 Axes>

In [10]:
env.close()
if eval_env is not None:
    eval_env.close()